# V3 E1 Embedding Smoke

Synthetic en-US model-loading smoke for the V3 sentence-embedding cycle.

In [ ]:
from pathlib import Path
import shutil
import sys
import zipfile

input_root = Path('/kaggle/input/cci-v3-e1-bundle')
work_root = Path('/kaggle/working')
bundle_root = work_root / 'v3_e1_bundle'
zip_path = next(input_root.rglob('cci-v3-e1-bundle.zip'))
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(bundle_root)
sys.path.insert(0, str(bundle_root / 'src'))
print(f'Code bundle staged at {bundle_root}')


In [ ]:
from consumer_complaint_intelligence import v3_kaggle as v3

staged_root = v3.stage_v3_e1_code(bundle_root, work_root)
print(f'Bundle manifest hashes verified at {staged_root}')
config_path = bundle_root / 'config' / 'v3_e1_execution.json'
config = v3.load_v3_e1_config(config_path)
print(f"Loaded V3 E1 config status={config['status']}")
v3.require_v3_e1_approval(config)
print('Execution approval gate passed')


In [ ]:
gpu_report = v3.assert_v3_e1_t4_gpu(config)
print(f"T4 GPU gate passed: {gpu_report['device_name']}")


In [ ]:
import subprocess

pin = config['package_policy']['sentence-transformers']
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    f'sentence-transformers=={pin}',
    '--no-deps',
])
v3.assert_v3_e1_sentence_transformers_pin(config)
print(f'sentence-transformers=={pin} ready without dependency changes')

output_path = Path('/kaggle/working/v3_e1_smoke.json')
result = v3.run_v3_e1_smoke(
    config,
    output_path=output_path,
    progress=print,
)
print(f"Wrote aggregate V3 E1 output to {output_path}")
print(f"Models completed: {len(result['models'])}")
